# 33. 분할 정복과 매개 변수 탐색

경시 · 문제 161–165

**사용법**
1. 바로 아래 **🔧 채점기 준비** 셀을 먼저 실행합니다(▶).
2. 각 문제의 `%%writefile` 셀에 코드를 작성하고 실행하면 파일로 저장됩니다.
3. 그 아래 `check(번호)` 셀을 실행하면 채점됩니다.
4. 직접 입력을 넣어 실행해 보려면 새 셀에 `%run p001.py`처럼 입력합니다.

> 정답과 해설은 위키독스 본문의 접힌 상자에 있습니다. 충분히 고민한 뒤에 펼쳐 보세요.

> 제3부의 시간 제한은 Colab 기준입니다. 숨은 테스트에는 제한 범위의 큰 입력이 들어 있으므로, 효율이 낮은 풀이는 '시간 초과'가 납니다.

In [ ]:
#@title 🔧 채점기 준비 (먼저 실행하세요)
TESTS = {161: [['5\n3 1 2 5 4', '3', True, None], ['4\n2 2 1 1', '4', True, None], ['1\n5', '0', False, None], ['5\n5 4 3 2 1', '10', False, None], [None, '10014320172', False, 'import random\nrandom.seed(114)\nn = 200000\nprint(n)\nprint(*[random.randint(-10**9, 10**9) for _ in range(n)])'], [None, '19999900000', False, 'import random\nrandom.seed(115)\nn = 200000\nprint(n)\nprint(*range(n, 0, -1))']], 162: [['4 11\n802 743 457 539', '200', True, None], ['1 1\n2147483647', '2147483647', False, None], ['2 2\n1 1', '1', False, None], ['3 5\n10 10 10', '5', False, None], [None, '107469', False, 'import random\nrandom.seed(116)\nk = 100000\nprint(k, 10**9)\nprint(*[random.randint(10**4, 2**31 - 1) for _ in range(k)])']], 163: [['5 3\n1 2 8 4 9', '3', True, None], ['2 2\n0 1000000000', '1000000000', False, None], ['4 4\n1 2 3 4', '1', False, None], ['6 2\n5 1 9 3 7 2', '8', False, None], [None, '996072', False, 'import random\nrandom.seed(117)\nn = 200000\nprint(n, 1000)\nprint(*random.sample(range(10**9 + 1), n))']], 164: [['10', '55', True, None], ['1000', '517691607', True, None], ['0', '0', False, None], ['1', '1', False, None], ['2', '1', False, None], ['90', '210345902', False, None], ['1000000000000000000', '209783453', False, None], ['999999999999999999', '470273943', False, None], ['2000000016', '0', False, None]], 165: [['3\n7', '6', True, None], ['4\n16', '16', True, None], ['1\n1', '1', False, None], ['2\n2', '2', False, None], ['2\n3', '2', False, None], ['100000\n1', '1', False, None], ['100000\n10000000000', '10000000000', False, None], ['100000\n5000000000', '1866871600', False, None], ['99999\n123456789', '16709626', False, None]]}
TL = {161: 3, 162: 2, 163: 3, 164: 1, 165: 3}
import subprocess, sys, pathlib, hashlib

_GEN_CACHE = {}

def _norm(s):
    return "\n".join(l.rstrip() for l in s.strip("\n").splitlines()).strip("\n")

def _input_of(inp, gen):
    if gen is None:
        return inp
    if gen not in _GEN_CACHE:
        _GEN_CACHE[gen] = subprocess.run([sys.executable, "-c", gen], capture_output=True, text=True).stdout
    return _GEN_CACHE[gen]

def check(pid):
    """p{번호}.py 를 모든 테스트로 실행해 채점합니다."""
    path = pathlib.Path(f"p{pid:03d}.py")
    if not path.exists():
        print(f"먼저 위의 %%writefile p{pid:03d}.py 셀을 실행하세요."); return
    limit = TL[pid]
    ok = 0
    for i, (inp, exp, shown, gen) in enumerate(TESTS[pid], 1):
        tag = f"예제 {i}" if shown else f"숨은 테스트 {i}"
        data = _input_of(inp, gen)
        try:
            r = subprocess.run([sys.executable, str(path)], input=data.rstrip("\n") + "\n",
                               capture_output=True, text=True, timeout=limit)
        except subprocess.TimeoutExpired:
            print(f"⏱ {tag}: 시간 초과 ({limit}초)"); continue
        got = _norm(r.stdout)
        if exp.startswith("sha256:"):
            got_cmp = "sha256:" + hashlib.sha256(got.encode()).hexdigest()
        else:
            got_cmp = got
        if r.returncode:
            last = r.stderr.strip().splitlines()[-1] if r.stderr.strip() else "알 수 없는 오류"
            print(f"💥 {tag}: 실행 오류 → {last}")
        elif got_cmp == exp:
            ok += 1; print(f"✅ {tag}")
        elif shown:
            print(f"❌ {tag}\n   입력: {inp!r}\n   기대: {exp!r}\n   실제: {got!r}")
        elif gen is not None:
            print(f"❌ {tag}  (큰 입력)")
        else:
            print(f"❌ {tag}  (입력: {inp[:60]!r})")
    total = len(TESTS[pid])
    print("─" * 30)
    print(f"🎉 {pid:03d}번 통과! ({ok}/{total})" if ok == total else f"{ok}/{total} 통과 — 다시 도전해 보세요.")
print('채점기 준비 완료!')

## 161. 역순쌍과 병합 정렬

길이 N의 수열에서 i < j이고 aᵢ > aⱼ인 쌍 (i, j)의 개수를 구하십시오.

**입력** 첫째 줄에 N, 둘째 줄에 N개의 정수가 주어집니다.

**출력** 역순쌍의 개수를 출력합니다.

**제한** 1 ≤ N ≤ 2×10⁵, |aᵢ| ≤ 10⁹, 시간 제한 3초

**예제 1**
입력
```
5
3 1 2 5 4
```
출력
```
3
```

**예제 2**
입력
```
4
2 2 1 1
```
출력
```
4
```

In [ ]:
%%writefile p161.py
# 여기에 코드를 작성하세요


In [ ]:
check(161)

## 162. 목재 재단

길이가 제각각인 통나무 K개를 모두 같은 정수 길이 L로 잘라 N조각 이상을 만들려 합니다(남는 토막은 버립니다). 가능한 L의 최댓값을 구하십시오. L = 1이면 언제나 가능하도록 입력이 주어집니다.

**입력** 첫째 줄에 K와 N, 둘째 줄에 K개의 통나무 길이가 주어집니다.

**출력** L의 최댓값을 출력합니다.

**제한** 1 ≤ K ≤ 10⁵, 1 ≤ N ≤ 10¹¹, 1 ≤ 길이 ≤ 2³¹ − 1, 시간 제한 2초

**예제 1**
입력
```
4 11
802 743 457 539
```
출력
```
200
```

In [ ]:
%%writefile p162.py
# 여기에 코드를 작성하세요


In [ ]:
check(162)

## 163. 등대지기

수직선 위의 서로 다른 N개 위치 가운데 C곳을 골라 등대를 세울 때, 가장 가까운 두 등대 사이 거리의 최댓값을 구하십시오.

**입력** 첫째 줄에 N과 C, 둘째 줄에 N개의 위치가 주어집니다.

**출력** 가장 가까운 두 등대 사이 거리의 최댓값을 출력합니다.

**제한** 2 ≤ C ≤ N ≤ 2×10⁵, 0 ≤ 위치 ≤ 10⁹, 시간 제한 3초

**예제 1**
입력
```
5 3
1 2 8 4 9
```
출력
```
3
```

In [ ]:
%%writefile p163.py
# 여기에 코드를 작성하세요


In [ ]:
check(163)

## 164. 거대한 피보나치

n이 주어질 때 F(n)을 1,000,000,007로 나눈 나머지를 구하십시오. F(0) = 0, F(1) = 1입니다.

**입력** 첫째 줄에 n이 주어집니다.

**출력** F(n) mod 1,000,000,007을 출력합니다.

**제한** 0 ≤ n ≤ 10¹⁸, 시간 제한 1초

**예제 1**
입력
```
10
```
출력
```
55
```

**예제 2**
입력
```
1000
```
출력
```
517691607
```

In [ ]:
%%writefile p164.py
# 여기에 코드를 작성하세요


In [ ]:
check(164)

## 165. 곱셈표의 K번째 수

N×N 곱셈표(i행 j열의 값은 i × j)의 모든 수를 오름차순으로 늘어놓았을 때 K번째 수를 구하십시오.

**입력** 첫째 줄에 N, 둘째 줄에 K가 주어집니다.

**출력** K번째 수를 출력합니다.

**제한** 1 ≤ N ≤ 10⁵, 1 ≤ K ≤ N², 시간 제한 3초

**예제 1**
입력
```
3
7
```
출력
```
6
```

**예제 2**
입력
```
4
16
```
출력
```
16
```

In [ ]:
%%writefile p165.py
# 여기에 코드를 작성하세요


In [ ]:
check(165)